In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json
import os
from collections import Counter
from statistics import mean

In [3]:
def analyze_jsonl(filepath):
    print(f"Analyzing: {filepath} ...\n")

    if not os.path.exists(filepath):
        print("Error: File not found.")
        return

    total_lines = 0
    malformed_lines = 0
    ids = []

    # query_stats tracks the types: KEYWORD, NATURAL, SEMANTIC
    query_type_counts = Counter()
    queries_per_row = []

    # To track distinct reasoning length or empty fields
    empty_reasoning_count = 0

    with open(filepath, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            total_lines += 1

            try:
                data = json.loads(line)

                # 1. Check ID
                if 'id' in data:
                    ids.append(str(data['id']))
                else:
                    print(f"[Warning] Line {line_num}: Missing 'id' field")

                # 2. Analyze Queries
                generated = data.get('generated_queries', [])

                # Check if it's a list (expected) or something else
                if isinstance(generated, list):
                    queries_per_row.append(len(generated))

                    for q in generated:
                        q_type = q.get('type', 'UNKNOWN')
                        query_type_counts[q_type] += 1

                        # Check specific logic for SEMANTIC types
                        if q_type == 'SEMANTIC':
                            reasoning = q.get('reasoning', '')
                            if not reasoning:
                                empty_reasoning_count += 1
                else:
                    print(f"[Warning] Line {line_num}: 'generated_queries' is not a list.")

            except json.JSONDecodeError:
                malformed_lines += 1
                print(f"[Error] Line {line_num}: Invalid JSON")

    # --- REPORTING ---
    print("="*40)
    print("       ANALYSIS REPORT")
    print("="*40)

    if total_lines == 0:
        print("File is empty.")
        return

    # Basic Stats
    unique_ids = set(ids)
    duplicate_count = len(ids) - len(unique_ids)

    print(f"Total Lines Processed:    {total_lines}")
    print(f"Malformed JSON Lines:     {malformed_lines}")
    print(f"Unique IDs found:         {len(unique_ids)}")
    print(f"Duplicate IDs:            {duplicate_count}")

    if ids:
        # Try to sort numerically if possible, otherwise string sort
        try:
            sorted_ids = sorted(ids, key=int)
            print(f"ID Range:                 {sorted_ids[0]} -> {sorted_ids[-1]}")
        except:
            print(f"First ID:                 {ids[0]}")
            print(f"Last ID:                  {ids[-1]}")

    print("-" * 40)
    print("QUERY STATISTICS")
    print("-" * 40)

    if queries_per_row:
        print(f"Avg Queries per Row:      {mean(queries_per_row):.2f}")
        print(f"Min Queries in a Row:     {min(queries_per_row)}")
        print(f"Max Queries in a Row:     {max(queries_per_row)}")
    else:
        print("No queries found.")

    print("\nBreakdown by Type:")
    for q_type, count in query_type_counts.most_common():
        print(f"  - {q_type:<10}: {count}")

    if empty_reasoning_count > 0:
        print(f"\n[Warning] {empty_reasoning_count} SEMANTIC queries represent missing 'reasoning'.")

    # Validation Alert
    expected_total_queries = total_lines * 3
    actual_total_queries = sum(queries_per_row)

    print("-" * 40)
    if actual_total_queries == expected_total_queries:
        print("✅ SUCCESS: All rows have exactly 3 queries.")
    else:
        print(f"⚠️  WARNING: Expected {expected_total_queries} queries, found {actual_total_queries}.")
        print("    Check 'Min/Max Queries in a Row' above.")

In [4]:
analyze_jsonl("/content/drive/MyDrive/KLTN_Project/Datasets/queries_triples.jsonl")

Analyzing: /content/drive/MyDrive/KLTN_Project/Datasets/queries_triples.jsonl ...

       ANALYSIS REPORT
Total Lines Processed:    2692
Malformed JSON Lines:     0
Unique IDs found:         2692
Duplicate IDs:            0
ID Range:                 0 -> 26644
----------------------------------------
QUERY STATISTICS
----------------------------------------
Avg Queries per Row:      3.00
Min Queries in a Row:     3
Max Queries in a Row:     3

Breakdown by Type:
  - KEYWORD   : 2692
  - NATURAL   : 2692
  - SEMANTIC  : 2692
----------------------------------------
✅ SUCCESS: All rows have exactly 3 queries.


# Using Pandas

In [5]:
import pandas as pd

def read_jsonl_pd(input_file: str):
    df = pd.read_json(input_file, lines=True)
    return df

query_triples_df = read_jsonl_pd("/content/drive/MyDrive/KLTN_Project/Datasets/queries_triples.jsonl")
query_triples_df.head()

,id,generated_queries
0,0,[{'query': 'Phó Thủ tướng Trần Hồng Hà chúc mừ...
1,1,[{'query': 'Tô Văn Hải đổ chất thải rắn chôn l...
2,2,[{'query': 'SAWACO tạm ngưng nước 25-3 22 giờ'...
3,3,[{'query': 'CLB NABC mùa 4 gây quỹ không đủ 10...
4,4,[{'query': 'ILA hỗ trợ học sinh miễn phí kiểm ...


In [6]:
query_triples_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2692 entries, 0 to 2691
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 2692 non-null   int64 
 1   generated_queries  2692 non-null   object
dtypes: int64(1), object(1)
memory usage: 42.2+ KB


In [8]:
query_triples_df['id'].max()

26644

In [10]:
query_triples_df[-10:]

,id,generated_queries
2682,2688,[{'query': 'Pháp Hà Lan toàn thắng sân nhà 4 c...
2683,2689,[{'query': 'Ban Chỉ đạo quốc gia hội nhập quốc...
2684,2690,[{'query': 'Quảng Ngãi ấp Hậu Lân Bà Điểm số n...
2685,2691,[{'query': 'Pháp khó khăn tổn thất lục đục sau...
2686,2692,[{'query': 'Volodymyr Zelensky 20 UAV tự sát t...
2687,2693,[{'query': 'diễn giả đến trễ trường không chỗ ...
2688,2694,[{'query': 'TikTok học tiếng Anh trào lưu sai ...
2689,2695,[{'query': 'Tổng thống UAE bổ nhiệm con trai c...
2690,2696,[{'query': 'quan chức ASEAN World Cup 2011 Lom...
2691,26644,[{'query': 'thai phụ 50 tuổi mang thai tự nhiê...


In [13]:
query_triples_df[-1:]

,id,generated_queries
2691,26644,[{'query': 'thai phụ 50 tuổi mang thai tự nhiê...


# Original dataset

In [11]:
original_data = pd.read_csv("/content/drive/MyDrive/KLTN_Project/Datasets/vifc.csv")
original_data

,Unnamed: 0,document,evidence,query,id
0,0,"(Chinhphu.vn) - Đây là mong muốn, gửi gắm của ...","Thay mặt Chính phủ, Thủ tướng Chính phủ, Phó T...","Phó Thủ tướng Trần Hồng Hà thay mặt Chính phủ,...",0
1,1,"Ngày 24/3, Cơ quan Cảnh sát điều tra Công an t...",Tô Văn Hải đã có hành vi cho phép người khác đ...,Hành vi của Tô Văn Hải là cho phép người khác ...,1
2,2,(PLO)- Theo Tổng Công ty Cấp nước Sài Gòn (SAW...,SAWACO thông báo tạm ngưng cung cấp nước để th...,SAWACO thông báo tạm ngưng cung cấp nước để th...,2
3,3,"Với khoảng 200 thành viên, UEF Warm Hugs Club ...",Là chương trình lớn nhất của CLB trong năm nên...,"CLB luôn chuẩn bị rất kỹ lưỡng, chỉn chu chươn...",3
4,4,"Đi qua 2 năm dịch bệnh đầy thử thách, nhưng ch...","Bé được tham gia kiểm tra trình độ đầu vào, tư...","ILA tiếp nhận và hỗ trợ học sinh miễn phí, Bé ...",4
...,...,...,...,...,...
34806,34806,"Vợ chồng Alicia Gabriela, giám đốc marketing n...",Với các trường quốc tế ở TP HCM đã công bố tuy...,Bậc mầm non ở trường hệ quốc tế tại TP HCM có ...,34806
34807,34807,"Vợ chồng Alicia Gabriela, giám đốc marketing n...","""Gia đình họ sau đó tìm một trường quốc tế chỉ...",Học phí gần 500 triệu mỗi năm cho một trường q...,34807
34808,34808,Theo thông báo được đăng trên tài khoản Wechat...,"Số lượng cặp kết hôn năm 2022 là 6,8 triệu, th...","Năm 2022, có 6,8 triệu cặp kết hôn theo số liệ...",34808
34809,34809,Học sinh trung học Wu Yuhan luôn thích đi xe đ...,Hai bố con cùng nhau lên đường vào 31/7. Họ đế...,"Sau khoảng 9 ngày, hai cha con đã đặt chân đến...",34809


In [12]:
original_data[original_data['Unnamed: 0'] != original_data['id']]

,Unnamed: 0,document,evidence,query,id
